# SmolVLA LIBERO-Object Evaluation

**Checkpoint**: `dennywu2966/smolvla-libero-object-lora` (20k steps, loss ~0.134)

**Key facts** (verified locally):
- `config.json` already has `use_peft=True` — no extra CLI flag needed
- Normalizer stats are 8D (matches LiberoProcessorStep eef_pos+axisangle+gripper)
- rename_map (image→camera1, image2→camera2) is embedded in `policy_preprocessor.json`


In [ ]:
# Cell 0: Install — explicit versions matching training setup
!pip install -q "lerobot[smolvla,peft,libero]>=0.4.3" "peft>=0.18.0"

import os, subprocess, sys
os.environ["MUJOCO_GL"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Verify LIBERO is installed
try:
    import libero  # noqa
    print("LIBERO: OK")
except ImportError as e:
    print(f"LIBERO missing: {e}")
    print("Run: pip install libero")
    raise


In [ ]:
# Cell 1: Verify checkpoint integrity
from huggingface_hub import hf_hub_download
from peft import PeftConfig
import json

LORA_REPO = "dennywu2966/smolvla-libero-object-lora"

peft_cfg = PeftConfig.from_pretrained(LORA_REPO)
print(f"base_model: {peft_cfg.base_model_name_or_path}")
print(f"peft_type: {peft_cfg.peft_type}, r={peft_cfg.r}")
assert peft_cfg.base_model_name_or_path == "lerobot/smolvla_base", "Wrong base model!"

cfg_path = hf_hub_download(LORA_REPO, "config.json")
with open(cfg_path) as f:
    cfg = json.load(f)
print(f"use_peft in config.json: {cfg.get('use_peft')}  (should be True)")
assert cfg.get("use_peft") is True, "use_peft not True in config.json!"

print("Checkpoint OK — ready for eval")


In [ ]:
# Cell 2: Evaluate SmolVLA-LoRA fine-tuned
# use_peft=True is already in config.json — read automatically by lerobot-eval
import subprocess, time, os

LORA_REPO = "dennywu2966/smolvla-libero-object-lora"
eval_env = {**os.environ, "MUJOCO_GL": "egl", "TOKENIZERS_PARALLELISM": "false"}

print(f"Evaluating: {LORA_REPO}")
print("Expected: ~2-3 hours for 20 episodes × 10 tasks")

start = time.time()
proc = subprocess.Popen([
    "lerobot-eval",
    f"--policy.path={LORA_REPO}",
    "--policy.device=cuda",
    "--env.type=libero",
    "--env.task=libero_object",
    "--eval.n_episodes=20",
    "--eval.batch_size=1",
], env=eval_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

ft_output = []
for line in proc.stdout:
    print(line, end="", flush=True)
    ft_output.append(line)

proc.wait(timeout=14400)  # 4h hard timeout
elapsed = time.time() - start
print(f"\nExit: {proc.returncode} | Time: {elapsed/3600:.2f}h")

if proc.returncode != 0:
    print("\n=== EVAL FAILED — last 30 lines ===")
    print("".join(ft_output[-30:]))


In [ ]:
# Cell 3: Evaluate official HuggingFaceVLA/smolvla_libero (upper bound reference)
import subprocess, time, os

OFFICIAL_REPO = "HuggingFaceVLA/smolvla_libero"
eval_env = {**os.environ, "MUJOCO_GL": "egl", "TOKENIZERS_PARALLELISM": "false"}

print(f"Evaluating official reference: {OFFICIAL_REPO}")

start = time.time()
proc = subprocess.Popen([
    "lerobot-eval",
    f"--policy.path={OFFICIAL_REPO}",
    "--policy.device=cuda",
    "--env.type=libero",
    "--env.task=libero_object",
    "--eval.n_episodes=20",
    "--eval.batch_size=1",
], env=eval_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

official_output = []
for line in proc.stdout:
    print(line, end="", flush=True)
    official_output.append(line)

proc.wait(timeout=14400)
elapsed = time.time() - start
print(f"\nExit: {proc.returncode} | Time: {elapsed/3600:.2f}h")


In [ ]:
# Cell 4: Parse results from outputs/ directory
import glob, json, re

def parse_eval_results(output_lines, label):
    """Parse success rates from lerobot-eval stdout."""
    results = {}
    for line in output_lines:
        # lerobot-eval prints: "pc_success: 0.75" or "success_rate: 0.75"
        m = re.search(r"(pc_success|avg_sum_reward|success).*?([\d.]+)", line)
        if m:
            print(f"[{label}] {line.rstrip()}")
    # Also check result files
    files = sorted(glob.glob("outputs/eval/*/eval_info.json"))
    print(f"\nResult files: {files}")
    for f in files:
        with open(f) as fh:
            data = json.load(fh)
        print(f"\n--- {f} ---")
        overall = data.get("overall", {})
        print(f"pc_success: {overall.get('pc_success', 'N/A')}")
        print(f"avg_sum_reward: {overall.get('avg_sum_reward', 'N/A')}")
        results[f] = data
    return results

print("=== SmolVLA-LoRA results ===")
ft_results = parse_eval_results(ft_output, "FT")

print("\n=== Official SmolVLA results ===")
official_results = parse_eval_results(official_output, "Official")


In [ ]:
# Cell 5: Final comparison table
import sys
sys.path.insert(0, "/kaggle/working/vlm-vla/src")
from vlm_vla.eval_engine import EvalReport, TaskResult, compare_reports

# --- Fill these from Cell 4 output ---
# ft_success_rate = 0.XX  # from SmolVLA-LoRA eval
# official_success_rate = 0.XX  # from official eval

# Template: replace with actual per-task results from Cell 4
# Per-task data available in outputs/eval/*/eval_info.json
import json, glob
result_files = sorted(glob.glob("outputs/eval/*/eval_info.json"))

if len(result_files) >= 2:
    # Parse both result files
    with open(result_files[0]) as f: r0 = json.load(f)
    with open(result_files[1]) as f: r1 = json.load(f)
    print("FT overall:", r0.get("overall", {}))
    print("Official overall:", r1.get("overall", {}))

# Reference baselines
smolvla_zs = EvalReport(
    model_name="SmolVLA-ZeroShot",
    task_suite="libero_object",
    results=[TaskResult(f"task_{i}", 0.0, 20, 400.0, {"timeout": 20}) for i in range(10)],
)
openvla_ref = EvalReport(
    model_name="OpenVLA-7B-FT",
    task_suite="libero_object",
    results=[TaskResult(f"task_{i}", 0.884, 20, 120.0, {}) for i in range(10)],
)

# TODO: build smolvla_ft from Cell 4 per-task data, then:
# print(compare_reports(smolvla_ft, smolvla_zs, openvla_ref))

print("\nReference baselines:")
print(compare_reports(smolvla_zs, openvla_ref))


In [ ]:
# Cell 6: Save results and update eval_report.md
import glob, json

# Collect all eval output JSON
all_results = {}
for f in sorted(glob.glob("outputs/eval/*/eval_info.json")):
    with open(f) as fh:
        all_results[f] = json.load(fh)

with open("eval_comparison_v2.json", "w") as fh:
    json.dump(all_results, fh, indent=2)
print(f"Saved {len(all_results)} eval results to eval_comparison_v2.json")

# Download as Kaggle output artifact
import shutil
shutil.copy("eval_comparison_v2.json", "/kaggle/working/eval_comparison_v2.json")
print("Artifact ready at /kaggle/working/eval_comparison_v2.json")
